# Stage 1b — Configuration-type separation (tilt-rotor / tilt-wing / lift+cruise / ...)

**This is the real thesis question.** Shrouded-vs-open (notebook 11) was a warm-up
that validated the whole pipeline on a clean 2-class problem. This notebook points
the *same machinery* at your actual taxonomy.

Thin notebook — it only **imports**, **calls** `src/analysis.py`, and **displays**.

Prerequisites:
1. Add a configuration-type column to the labels Excel (e.g. `Configuration`,
   values like `tilt_rotor`, `tilt_wing`, `lift_cruise`, `multirotor`, `ducted`).
2. Set `data.label_cols: ["Configuration"]` (or your chosen name) in `config.yaml`
   and `data.config_type_col` to match.
3. Run `10_qc_embeddings.ipynb` first if embeddings aren't already on disk.

What this notebook answers:
1. **Class balance** — how many images per configuration (small/imbalanced classes
   get dropped automatically below a minimum count).
2. **Multinomial probe** — can a linear classifier tell configurations apart, and
   how much better than chance (`1/n_classes`)?
3. **Confusion matrix** — *which* configurations get mistaken for which. This is
   the most informative output: it tells you where frozen DINOv2 already separates
   designs, and where fine-tuning will need to do the work.
4. **Class similarity matrix** — the same question without a classifier (purely
   geometric): are two configurations close together in embedding space?

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config_loader import load_config
from src import data, embeddings, analysis

cfg = load_config()
SEED = cfg['analysis']['seed']
CONFIG_COL = cfg['data'].get('config_type_col', 'Configuration')

result = embeddings.load_embeddings(cfg)
arrays, metadata = result['arrays'], result['metadata']

labels = data.load_labels(cfg)
if CONFIG_COL not in labels.columns:
    raise KeyError(
        f"'{CONFIG_COL}' not found in labels. Add the column to the Excel and to "
        f"config.yaml -> data.label_cols, then re-run data.load_labels."
    )

y = analysis.multiclass_labels(metadata, labels, col=CONFIG_COL)
print(f"figures total       : {len(y)}")
print(f"labelled (non-NaN)  : {int(y.notna().sum())}")
print('class counts (labelled only):')
print(y.value_counts())

## 1 — Class balance

Configurations with very few examples will be **dropped automatically** by
`min_per_class` (default 4) in the probe/confusion functions below — you'll see
`n_classes` and `n` in the output shrink accordingly. If an important class is
being dropped, that's a labelling-coverage problem to fix before trusting the rest.

In [ ]:
MIN_PER_CLASS = 4   # raise once you have more images per class

vc = y.value_counts()
print('classes below MIN_PER_CLASS (will be excluded from probes):')
print(vc[vc < MIN_PER_CLASS] if (vc < MIN_PER_CLASS).any() else '  (none)')

## 2 — Multinomial probe (per matrix)

Same discipline as the binary case: PCA-50, 5-fold (auto-capped at the smallest
class size) stratified CV, **macro-F1** + **balanced accuracy** + **macro
one-vs-rest AUC**, with a **permutation p-value** on balanced accuracy.

`chance` = `1/n_classes` (e.g. 0.20 for 5 classes) — balanced_acc must clear that,
not 0.5. Compare across the 6 (layer, pooling) matrices to find which representation
best separates configurations — the same logic as `structure_report` in notebook 11.

In [ ]:
rows = []
for (L, pool), X in sorted(arrays.items()):
    try:
        r = analysis.multiclass_probe(X, y.to_numpy(), min_per_class=MIN_PER_CLASS, seed=SEED)
    except ValueError as e:
        print(f"layer{L}_{pool}: skipped ({e})")
        continue
    rows.append({"layer": L, "pooling": pool, **{k: r[k] for k in
                ['n', 'n_classes', 'macro_f1', 'balanced_acc', 'macro_auc', 'chance', 'p_value']}})

config_report = pd.DataFrame(rows)
out = Path(cfg['paths']['output_dir']) / 'config_type_report.csv'
config_report.to_csv(out, index=False); print('wrote', out)
config_report

**How to read this table:** find the row with the highest `balanced_acc` /
`macro_auc` relative to its `chance` level, and the smallest `p_value`. That's your
best frozen representation for the configuration taxonomy — likely not the same
layer that won for shrouded/open, since this is a harder, more fine-grained
question.

## 3 — Confusion matrix (best matrix)

Row = true configuration, column = predicted. The diagonal is what the model gets
right; **off-diagonal mass is the interesting part** — it shows exactly which pairs
of configurations the frozen model conflates (e.g. tilt-rotor vs tilt-wing are
visually similar in many patent views, so expect some confusion there).

In [ ]:
# Pick the matrix with the best balanced_acc from the table above.
best_row = config_report.loc[config_report['balanced_acc'].idxmax()]
best_key = (int(best_row['layer']), best_row['pooling'])
print('best matrix:', best_key)

cm = analysis.class_confusion_matrix(arrays[best_key], y.to_numpy(),
                                     min_per_class=MIN_PER_CLASS, seed=SEED)
cm

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm.values, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(cm.columns))); ax.set_xticklabels(cm.columns, rotation=45, ha='right')
ax.set_yticks(range(len(cm.index))); ax.set_yticklabels(cm.index)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'Configuration confusion matrix — layer{best_key[0]}_{best_key[1]}')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f'{cm.values[i,j]:.2f}', ha='center', va='center',
                color='white' if cm.values[i, j] > 0.5 else 'black')
fig.colorbar(im, label='row-normalised fraction')
fig.tight_layout()
png = Path(cfg['paths']['output_dir']) / 'config_confusion_matrix.png'
fig.savefig(png, dpi=130, bbox_inches='tight'); print('saved', png)
plt.show()

## 4 — Class similarity matrix (unsupervised, no classifier)

Mean pairwise cosine similarity **within** each class (diagonal) and **between**
classes (off-diagonal) — purely geometric, no probe involved. High off-diagonal
similarity between two classes is the same story as confusion-matrix mistakes, seen
from a different angle: those configurations sit close together in embedding space.

In [ ]:
sim = analysis.class_similarity_matrix(arrays[best_key], y.to_numpy(), min_per_class=MIN_PER_CLASS)
sim

## Verdict

Summarize, once real labels are in: which configurations does frozen DINOv2 already
separate well (high diagonal, low confusion), and which pairs are entangled
(high off-diagonal similarity / confusion)? The entangled pairs are exactly what
**fine-tuning** needs to fix — this notebook becomes your *before* picture.